# Starter — a notebook that can fail

*Replace this notebook. It exists to show the house rules working, not to be kept.*

The rule this demonstrates is the one in this folder's README:

> Make notebooks assert. A notebook that only prints cannot fail, which makes it
> documentation pretending to be a check.

And the house rule beneath it, which is a real bug this programme hit:

> If a scenario labelled "worst" outperforms one labelled "best", the parameters
> and the formula disagree about which way "worse" points.


In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine

# The PG* variables are already set by the dev container. Nothing here
# holds a host or a password, so this notebook is the same on every machine.
engine = create_engine(
    'postgresql+psycopg://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}'.format(**os.environ)
)


## The exposure

Who holds critical knowledge, and how close are they to leaving? That is the
whole business case in one join.


In [ ]:
people = pd.read_sql(
    '''select p.name, p.role, d.name as department, p.retirement_risk_date,
              p.is_steward
         from person p join department d on d.id = p.department_id
        order by p.retirement_risk_date nulls last''',
    engine,
)
people


In [ ]:
import numpy as np

horizon = pd.Timestamp.now(tz='UTC')
people['years_to_risk'] = (
    pd.to_datetime(people.retirement_risk_date, utc=True) - horizon
).dt.days / 365.25
people[['name', 'role', 'department', 'years_to_risk', 'is_steward']]


## Assertions, not prints

Each of these would fail loudly if the data or the reasoning moved. That is the
difference between a check and a report.


In [ ]:
# 1. The frame is not silently empty. An empty frame satisfies almost every
#    aggregate you could write about it, which is how a broken query passes.
assert len(people) > 0, 'no people returned; the join or the seed is broken'

# 2. Anyone flagged as at risk has a date to be at risk by.
at_risk = people[people.years_to_risk.notna()]
assert (at_risk.years_to_risk > -50).all(), 'a retirement risk date sits in the deep past'

# 3. The ordering means what its name says -- the failure the house rule names.
ordered = people.sort_values('years_to_risk')
soonest, latest = ordered.years_to_risk.iloc[0], ordered.years_to_risk.dropna().iloc[-1]
assert soonest <= latest, (
    f'"soonest" ({soonest:.1f}y) is later than "latest" ({latest:.1f}y); '
    'the sort and the label disagree about which way time points'
)
print(f'{len(people)} people, {at_risk.is_steward.sum()} stewards, '
      f'nearest risk in {soonest:.1f} years')


## Before you commit

`jupyter nbconvert --clear-output --inplace <notebook>.ipynb`

Stored outputs turn every run into a diff and put whatever the cells printed
into the repository history, permanently.
